#### SQLite is a lightweight disk-based database that doesn't require a separate server process.

In [1]:
# Create a db of Monty Python movies
#
# Create a new database and open a database connection to the "tutorial.db"
# database (or create it if it doesn't exist).
import sqlite3

con = sqlite3.connect("tutorial.db")

# To execute SQL statements and queries, we need to create a database cursor
cur = con.cursor()

# Check the "sqlite_master" table whether "movie" table already exists
res = cur.execute("SELECT name FROM sqlite_master WHERE name='movie'")

if res.fetchone() is None:
    # Table doesn't exist so create it.
    # Create a db table "movie" with columns for title, year and review score.
    cur.execute("CREATE TABLE movie(title, year, score)")
else:
    print("Table already exists!")

In [2]:
# Add two rows of data as SQL literals
cur.execute(
    """
            INSERT INTO movie VALUES
            ('Monty Python and the Holy Grail', 1975, 8.2),
            ('And Now for Something Completely Different', 1971, 7.5)
            """
)

# NOTE: The changes have not yet been saved.
#       To commit the transaction and save the changes in the database:
con.commit()

In [ ]:
# Verify the data was inserted correctly.
# fetchall() returns all resulting rows.
res = cur.execute("SELECT score FROM movie")
res.fetchall()

[(8.2,), (7.5,)]

In [4]:
# Insert three more rows
data = [
    ("Monty Python Live at the Hollywood Bowl", 1982, 7.9),
    ("Monty Python's The Meaning of Life", 1983, 7.5),
    ("Monty Python's Life of Bryan", 1979, 8.0),
]

cur.executemany("INSERT INTO movie VALUES(?, ?, ?)", data)

# Remember to commit the transaction after executing INSERT
con.commit()

In [6]:
# Verify the new rows were inserted by iterating over the results of the query
for row in cur.execute("SELECT year, title FROM movie ORDER BY year"):
    print(row)

(1971, 'And Now for Something Completely Different')
(1975, 'Monty Python and the Holy Grail')
(1979, "Monty Python's Life of Bryan")
(1982, 'Monty Python Live at the Hollywood Bowl')
(1983, "Monty Python's The Meaning of Life")


In [ ]:
con.close()                                 # close the connection to the db
new_con = sqlite3.connect("tutorial.db")    # open a new connection
new_cur = new_con.cursor()                  # create a new cursor

# Order the results by descending score, so that the first record returned
# will be the one with the highest score.
res = new_cur.execute("SELECT title, year FROM movie ORDER BY score DESC")
# The rows returned are in the form of [(title, year), (title, year),...]
# so unpack the first tuple into title and year:
title, year = res.fetchone()
print(f"The hightest scoring Monty Python movie is {title!r}, released in {year}.")

new_con.close()

The hightest scoring Monty Python movie is 'Monty Python and the Holy Grail', released in 1975


#### Adapter and Converter Recipes

In [ ]:
import datetime
import sqlite3

# ADAPTERS
def adapt_date_iso(val: datetime.date):
    """Adapt datetime.date to ISO 8601 date: YYYY-MM-DD"""
    return val.isoformat()
def adapt_datetime_iso(val: datetime.datetime):
    """Adapt datetime.datetime to timezone-naive ISO 8601 date."""
    return val.replace(tzinfo=None).isoformat()
def adapt_datetime_epoch(val: datetime.datetime):
    """Adapt datetime.datetime to Unix timestamp"""
    return int(val.timestamp())

sqlite3.register_adapter(datetime.date,adapt_date_iso)
sqlite3.register_adapter(datetime.datetime,adapt_datetime_iso)
sqlite3.register_adapter(datetime.datetime,adapt_datetime_epoch)

# CONVERTERS
def convert_date(val: str):
    """Convert ISO 8601 date, to datetime.date object"""
    return datetime.date.fromisoformat(val.decode())
def convert_datetime(val:str):
    """Convert ISO 8601 datetime, to datetime.datetime object"""
    return datetime.datetime.fromisoformat(val.decode())
def convert_timestamp(val: str):
    """Convert Unix epoch timestamp to datetime.datetime object"""
    return datetime.datetime.fromtimestamp(int(val))

sqlite3.register_converter("date",convert_date)
sqlite3.register_converter("datetime",convert_datetime)
sqlite3.register_converter("timestamp",convert_timestamp)

#### How to use connection shorthand methods

In [ ]:
# Create and fill the table
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE lang(name, first_appeared)")
data = [
    ("C++", 1985),
    ("Objective-C", 1984),
]

con.executemany("INSERT INTO lang VALUES(?, ?)", data)

# Print the table contents
for row in con.execute("SELECT * FROM lang"):
    print(row)

# Delete rows
print("I just deleted", con.execute("DELETE FROM lang").rowcount, "rows")

con.close()

('C++', 1985)
('Objective-C', 1984)
I just deleted 2 rows


#### How to use the connection context manager

In [26]:
con = sqlite3.connect(":memory:")
sql = """
    CREATE TABLE lang(id    INTEGER     PRIMARY KEY,
                      name  VARCHAR     UNIQUE)
"""
con.execute(sql)

# Successful, con.commit() is called automatically afterwards
try:
    with con:
        con.execute("INSERT INTO lang(name) VALUES(?)", ("Python",))
except sqlite3.IntegrityError:
    print("Couldn't add Python twice")

# Connection object used as context manager, only commits or
# rollbacks transactions. The connection should be closed explicitly.
con.close()